In [1]:
"""
simulation.py

4 Node Random Mobility Simulation
---------------------------------
• One node starts at the center.
• Other nodes move randomly.
• Nodes try to maintain a minimum distance (bird-like separation).
• Collision is counted if two nodes come closer than COLLISION_DISTANCE.
• Can be imported by experiment.py or run directly.

Requirements:
    pip install pygame numpy
"""

import pygame
import random
import math
import numpy as np

# ------------------------------
# Simulation Parameters
# ------------------------------
NUM_NODES = 4
NODE_RADIUS = 10
SPEED = 2.5

SAFE_DISTANCE = 45
COLLISION_DISTANCE = 20

BACKGROUND = (245, 245, 245)
NODE_COLOR = (30, 144, 255)
CENTER_COLOR = (220, 30, 30)
TEXT_COLOR = (20, 20, 20)

FPS = 60


# ------------------------------
# Node Class
# ------------------------------
class Node:

    def __init__(self, x, y, center=False):

        self.position = np.array([x, y], dtype=float)

        angle = random.uniform(0, 2 * math.pi)

        self.velocity = np.array([
            math.cos(angle),
            math.sin(angle)
        ]) * SPEED

        self.center = center

    def move(self):

        self.position += self.velocity

    def keep_inside(self, width, height):

        if self.position[0] <= NODE_RADIUS:
            self.position[0] = NODE_RADIUS
            self.velocity[0] *= -1

        if self.position[0] >= width - NODE_RADIUS:
            self.position[0] = width - NODE_RADIUS
            self.velocity[0] *= -1

        if self.position[1] <= NODE_RADIUS:
            self.position[1] = NODE_RADIUS
            self.velocity[1] *= -1

        if self.position[1] >= height - NODE_RADIUS:
            self.position[1] = height - NODE_RADIUS
            self.velocity[1] *= -1

    def random_turn(self):

        angle = random.uniform(-0.15, 0.15)

        c = math.cos(angle)
        s = math.sin(angle)

        vx = self.velocity[0]
        vy = self.velocity[1]

        self.velocity[0] = vx * c - vy * s
        self.velocity[1] = vx * s + vy * c

        length = np.linalg.norm(self.velocity)

        if length != 0:
            self.velocity = self.velocity / length * SPEED


# ------------------------------
# Collision Avoidance
# ------------------------------
def separation(nodes):

    for i in range(len(nodes)):

        force = np.array([0.0, 0.0])

        for j in range(len(nodes)):

            if i == j:
                continue

            diff = nodes[i].position - nodes[j].position

            dist = np.linalg.norm(diff)

            if 0 < dist < SAFE_DISTANCE:

                force += diff / dist

        if np.linalg.norm(force) > 0:

            force = force / np.linalg.norm(force)

            nodes[i].velocity += force * 0.25

            length = np.linalg.norm(nodes[i].velocity)

            nodes[i].velocity = nodes[i].velocity / length * SPEED


# ------------------------------
# Collision Counter
# ------------------------------
def count_collisions(nodes):

    collisions = 0

    for i in range(len(nodes)):

        for j in range(i + 1, len(nodes)):

            d = np.linalg.norm(nodes[i].position - nodes[j].position)

            if d < COLLISION_DISTANCE:
                collisions += 1

    return collisions


# ------------------------------
# Draw Nodes
# ------------------------------
def draw(screen, nodes, collision_count, width, height):

    screen.fill(BACKGROUND)

    font = pygame.font.SysFont("Arial", 20)

    text = font.render(
        f"Collisions : {collision_count}",
        True,
        TEXT_COLOR
    )

    screen.blit(text, (10, 10))

    size_text = font.render(
        f"Window : {width} x {height}",
        True,
        TEXT_COLOR
    )

    screen.blit(size_text, (10, 35))

    # draw links
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):

            pygame.draw.line(
                screen,
                (180, 180, 180),
                nodes[i].position.astype(int),
                nodes[j].position.astype(int),
                1
            )

    # draw nodes
    for node in nodes:

        color = CENTER_COLOR if node.center else NODE_COLOR

        pygame.draw.circle(
            screen,
            color,
            node.position.astype(int),
            NODE_RADIUS
        )


# ------------------------------
# Main Simulation Function
# ------------------------------
def run_simulation(window_size=(500, 500),
                   simulation_time=20,
                   visualize=True):

    pygame.init()

    width, height = window_size

    if visualize:
        screen = pygame.display.set_mode(window_size)
        pygame.display.set_caption("Random Mobility Simulation")
    else:
        screen = pygame.Surface(window_size)

    clock = pygame.time.Clock()

    nodes = []

    # Center Node
    nodes.append(
        Node(width / 2, height / 2, center=True)
    )

    # Remaining Nodes
    for _ in range(NUM_NODES - 1):

        while True:

            x = random.randint(40, width - 40)
            y = random.randint(40, height - 40)

            ok = True

            for n in nodes:

                if np.linalg.norm(
                        np.array([x, y]) - n.position) < SAFE_DISTANCE:

                    ok = False
                    break

            if ok:
                nodes.append(Node(x, y))
                break

    total_collisions = 0

    frame_count = simulation_time * FPS

    running = True

    while running and frame_count > 0:

        frame_count -= 1

        for event in pygame.event.get():

            if event.type == pygame.QUIT:
                running = False

        separation(nodes)

        for node in nodes:

            node.random_turn()

            node.move()

            node.keep_inside(width, height)

        total_collisions += count_collisions(nodes)

        if visualize:

            draw(
                screen,
                nodes,
                total_collisions,
                width,
                height
            )

            pygame.display.flip()

        clock.tick(FPS)

    pygame.quit()

    return total_collisions


# ------------------------------
# Run Directly
# ------------------------------
if __name__ == "__main__":

    collisions = run_simulation(
        window_size=(500, 500),
        simulation_time=20,
        visualize=True
    )

    print("\nSimulation Finished")
    print("----------------------------")
    print("Total Collisions :", collisions)

pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html

Simulation Finished
----------------------------
Total Collisions : 0


In [2]:
"""
experiment.py

Runs the random mobility simulation for
5 different window sizes and plots

Window Size  vs  Total Collisions

Requirements
------------
pip install matplotlib pygame numpy

Place this file in the same folder as simulation.py
"""

import matplotlib.pyplot as plt
from simulation import run_simulation

# -------------------------------------------------
# Different Window Sizes
# -------------------------------------------------

WINDOW_SIZES = [
    (200, 200),
    (300, 300),
    (400, 400),
    (500, 500),
    (600, 600)
]

collisions = []
areas = []

print("=" * 60)
print(" RANDOM MOBILITY SIMULATION EXPERIMENT ")
print("=" * 60)

# -------------------------------------------------
# Run Experiment
# -------------------------------------------------

for size in WINDOW_SIZES:

    print(f"\nRunning Simulation for Window {size[0]} x {size[1]}")

    collision = run_simulation(
        window_size=size,
        simulation_time=20,     # seconds
        visualize=False         # True if you want animation every run
    )

    collisions.append(collision)
    areas.append(size[0] * size[1])

    print("Total Collisions :", collision)

print("\nExperiment Finished!")

# -------------------------------------------------
# Print Result Table
# -------------------------------------------------

print("\n")
print("-" * 45)
print(" Window Size\t\tCollisions")
print("-" * 45)

for s, c in zip(WINDOW_SIZES, collisions):
    print(f"{s[0]} x {s[1]}\t\t{c}")

# -------------------------------------------------
# Plot Graph
# -------------------------------------------------

window_labels = [
    "200x200",
    "300x300",
    "400x400",
    "500x500",
    "600x600"
]

plt.figure(figsize=(8, 5))

plt.plot(
    window_labels,
    collisions,
    marker="o",
    linewidth=2,
    markersize=8
)

plt.title("Window Size vs Collision Count")

plt.xlabel("Window Size")

plt.ylabel("Total Collisions")

plt.grid(True)

plt.tight_layout()

plt.savefig("collision_vs_window_size.png", dpi=300)

plt.show()

# -------------------------------------------------
# Relationship Analysis
# -------------------------------------------------

print("\n")
print("=" * 60)
print("FINAL OBSERVATION")
print("=" * 60)

for i in range(len(collisions) - 1):

    if collisions[i + 1] < collisions[i]:

        print(
            f"{WINDOW_SIZES[i]} -> {WINDOW_SIZES[i+1]} : "
            "Collision Decreased"
        )

    elif collisions[i + 1] > collisions[i]:

        print(
            f"{WINDOW_SIZES[i]} -> {WINDOW_SIZES[i+1]} : "
            "Collision Increased"
        )

    else:

        print(
            f"{WINDOW_SIZES[i]} -> {WINDOW_SIZES[i+1]} : "
            "No Change"
        )

print("\nOverall Relationship:")

if collisions[-1] < collisions[0]:

    print(
        "Larger Window Size  ==>  Fewer Collisions"
    )

elif collisions[-1] > collisions[0]:

    print(
        "Larger Window Size  ==>  More Collisions"
    )

else:

    print(
        "Window Size has little effect on collisions."
    )

print("=" * 60)

# -------------------------------------------------
# Optional: Save Results to File
# -------------------------------------------------

with open("results.txt", "w") as file:

    file.write("Window Size vs Collision\n")
    file.write("=============================\n")

    for s, c in zip(WINDOW_SIZES, collisions):
        file.write(f"{s[0]}x{s[1]} : {c}\n")

    file.write("\n")

    if collisions[-1] < collisions[0]:
        file.write("Conclusion: Larger windows produce fewer collisions.\n")
    elif collisions[-1] > collisions[0]:
        file.write("Conclusion: Larger windows produce more collisions.\n")
    else:
        file.write("Conclusion: Window size has minimal effect.\n")

print("\nResults saved to:")
print("  • collision_vs_window_size.png")
print("  • results.txt")

ModuleNotFoundError: No module named 'simulation'